# 🚀 Notebook 06 — Advanced Gradient Boosting Model Training Suite

<div style='background: linear-gradient(135deg, #f093fb 0%, #f5576c 100%); padding: 20px; border-radius: 10px; color: white; margin: 10px 0;'>

**Advanced GBDT Model Training Engine** — Fits high-capacity Gradient Boosted Decision Tree architectures (LightGBM, CatBoost, XGBoost) with GPU acceleration and early stopping.

</div>

| Property | Value |
|:---|:---|
| 🏗️ **Target** | `unit_sales` (log-transformed $\log(1 + y)$ during model fitting) |
| 📥 **Input** | `feature_store.parquet` (74 engineered features) |
| ⚡ **Hardware** | Automatic Hardware Detection: NVIDIA CUDA GPU (XGBoost/CatBoost) \| CPU Multi-Threading 12 Cores (LightGBM) |
| 📤 **Model Artifacts** | Saved to `03_Models/advanced_models/*.joblib` |
| 🖼️ **Plots Directory** | Saved to `output/06_model_training/` |
| 🤖 **Algorithms** | LightGBM (Leaf-wise), CatBoost (Ordered Boosting), XGBoost (Histogram/CUDA) |

---

### 📑 Table of Contents

| # | Section | Technical Purpose |
|:---:|:---|:---| 
| 1 | Environment Setup | Bootstrap project paths and Python environment |
| 2 | System Setup & Hardware Check | Detect NVIDIA CUDA GPU, CPU cores, load dataset & split |
| 2.1 | Data Load & Split | Chronological out-of-time train/val split & $\log(1+y)$ target scaling |
| 2.2 | Metric Evaluation Engine | Standardized RMSLE, RMSE, MAE, MAPE, $R^2$, and runtime tracking |
| 3 | LightGBM Regressor | Leaf-wise GBDT training with OpenMP multi-threading & feature importances |
| 4 | CatBoost Regressor | Ordered GBDT training with native GPU acceleration & feature importances |
| 5 | XGBoost Regressor | Histogram CUDA GBDT training with early stopping & feature importances |
| 6 | Advanced Leaderboard | Comparative benchmark leaderboard GBDTs vs Baseline Random Forest |
| 7 | Cleanup & Summary | Resource release and training artifact verification |


---
## 1️⃣ Environment Setup & Project Bootstrap

> **🎯 Purpose:** Ensure project root directory is added to Python `sys.path` so that project modules (`config.py`, `utils.py`) can be imported seamlessly from subfolders.

| Item | Description |
|:---|:---|
| **Input** | Current working directory |
| **Output** | `sys.path` updated with project root path |


In [ ]:
# TODO: Implement your code here


---
## 2️⃣ System Setup, Hardware Acceleration & Output Config

> **🎯 Purpose:** Import scientific packages, configure matplotlib aesthetics, and run **Automatic Hardware Detection** to query NVIDIA CUDA GPU and CPU multi-threading capabilities.

### ⚙️ How Automatic Hardware Detection Works
1. **XGBoost Test:** Tries to initialize `xgb.XGBRegressor(tree_method='hist', device='cuda')`. If CUDA fails, falls back to CPU multi-threading.
2. **CatBoost Test:** Tries to initialize `cb.CatBoostRegressor(task_type='GPU')`. If CUDA fails, falls back to CPU.
3. **LightGBM Strategy:** Configures `n_jobs=-1` to utilize all 12 CPU cores via OpenMP multi-threading (optimal for tabular data).

| Configuration | Path / Setting |
|:---|:---|
| **Output Reports & Plots** | `output/06_model_training/` |
| **Model Artifacts Registry** | `03_Models/advanced_models/` |


In [ ]:
# TODO: Implement your code here


### 📊 2.1 Dataset Load & Out-of-Time Chronological Split

> **🎯 Purpose:** Load observations from `feature_store.parquet` and split into training and validation sets strictly by date to preserve temporal order.

### ⚙️ Why Target Log Transformation $\log(1 + y)$?
- Demand sales exhibit heavy right skewness ($0$ to $500+$ units).
- Fitting models on $\log(1 + y)$ stabilizes variance, prevents large-sale outlier dominance, and directly optimizes the **RMSLE** objective metric.
- During prediction, forecasts are mapped back using $\exp(y_{pred}) - 1$.

| Dataset Split | Date Range | Size |
|:---|:---|:---|
| **Train Set** | Past history up to `2017-07-31` | ~1.95M rows |
| **Validation Set** | Holdout period: `2017-08-01` → `2017-08-15` (16 days) | ~50K rows |


In [ ]:
# TODO: Implement your code here


### 📊 2.2 Standardized Metric Evaluation Engine

> **🎯 Purpose:** Standardize metric calculation across all models (RMSLE, RMSE, MAE, MAPE, $R^2$) and track training/prediction runtimes.

| Metric | Formula | Interpretation |
|:---|:---|:---|
| **RMSLE** (Primary) | $\sqrt{\frac{1}{N}\sum (\log(1+y) - \log(1+\hat{y}))^2}$ | Penalizes relative percentage error |
| **RMSE** | $\sqrt{\frac{1}{N}\sum (y - \hat{y})^2}$ | Penalizes large absolute errors |
| **MAE** | $\frac{1}{N}\sum \|y - \hat{y}\|$ | Average absolute error units |
| **MAPE** | $\frac{100\%}{N}\sum \|\frac{y - \hat{y}}{y}\|$ | Percentage error on non-zero sales |


In [ ]:
# TODO: Implement your code here


---
## 3️⃣ Model 1 — LightGBM Regressor (Leaf-Wise Tree Growth)

> **🎯 Purpose:** Train LightGBM regressor using leaf-wise tree splitting (`num_leaves=63`) and early stopping (50 rounds) on validation loss.

### ⚙️ Model Hyperparameters & Export Artifacts
- **Hyperparameters:** `n_estimators=2000`, `learning_rate=0.03`, `num_leaves=63`, `subsample=0.8`, `colsample_bytree=0.8`.
- **Feature Importance Plot:** Saved to `output/06_model_training/01_lightgbm_feature_importance.png` and displayed inline.
- **Model Artifact:** Saved to `03_Models/advanced_models/lightgbm_model.joblib`.


In [ ]:
# TODO: Implement your code here


---
## 4️⃣ Model 2 — CatBoost Regressor (Ordered Boosting & GPU Support)

> **🎯 Purpose:** Train CatBoost regressor using ordered boosting and native NVIDIA CUDA GPU acceleration (`task_type='GPU'`).

### ⚙️ Model Hyperparameters & Export Artifacts
- **Hyperparameters:** `iterations=1500`, `depth=7`, `learning_rate=0.04`, `eval_metric='RMSE'`.
- **Feature Importance Plot:** Saved to `output/06_model_training/02_catboost_feature_importance.png` and displayed inline.
- **Model Artifact:** Saved to `03_Models/advanced_models/catboost_model.joblib`.


In [ ]:
# TODO: Implement your code here


---
## 5️⃣ Model 3 — XGBoost Regressor (Histogram CUDA GBDT)

> **🎯 Purpose:** Train XGBoost regressor using histogram-based binning (`tree_method='hist'`) and CUDA GPU acceleration (`device='cuda'`).

### ⚙️ Model Hyperparameters & Export Artifacts
- **Hyperparameters:** `n_estimators=1500`, `max_depth=7`, `learning_rate=0.04`, `subsample=0.8`, `colsample_bytree=0.8`.
- **Feature Importance Plot:** Saved to `output/06_model_training/03_xgboost_feature_importance.png` and displayed inline.
- **Model Artifact:** Saved to `03_Models/advanced_models/xgboost_model.joblib`.


In [ ]:
# TODO: Implement your code here


---
## 6️⃣ Advanced Models Leaderboard & Comparative Diagnostics

> **🎯 Purpose:** Compile training metrics into a styled leaderboard table, compare GBDTs against baseline Random Forest (`RMSLE = 0.2873`), and export JSON/CSV benchmark reports.

| Export File | Purpose |
|:---|:---|
| `output/06_model_training/advanced_models_results.csv` | Summary table with all metrics |
| `output/06_model_training/training_metrics.json` | Machine-readable metrics dictionary |
| `output/06_model_training/04_advanced_models_comparison.png` | Grouped bar chart comparing RMSLE and Training Runtimes |


In [ ]:
# TODO: Implement your code here


---

<div style='background: linear-gradient(135deg, #f093fb 0%, #f5576c 100%); padding: 15px; border-radius: 10px; color: white; text-align: center; margin: 10px 0;'>

**End of Notebook 06 — Advanced Model Training Suite** ✅

| Stage | Next Notebook |
|:---|:---|
| ✅ Model Training | → [07_model_evaluation.ipynb](./07_model_evaluation.ipynb) |

</div>
